# Taller: POO, Interfaces Gráficas y Bases de Datos (Supabase)

En esta lección vamos a crear una aplicación completa utilizando **Python**. Aprenderemos a conectar nuestro programa con **Supabase**, una alternativa moderna y de código abierto a Firebase (basada en PostgreSQL), y a crear una interfaz visual con **Tkinter**.

## 1. Definiendo nuestra Entidad (El Backend y POO)
Al igual que en otros sistemas, usamos Programación Orientada a Objetos para crear un molde (Clase) que representará a la información a guardar en la base de datos.

In [ ]:
class Usuario():
    lista=[]
    def __init__(self,name,age,pasword):
        self.nombre=name
        self.edad=age
        self.contra=pasword
        if self not in Usuario.lista:
            Usuario.lista.append(self)

    def mostrar_datos(self):
        return f"El usuario {self.nombre} tiene {self.edad} y su pasword es {self.contra}"
    
    @classmethod
    def mostrar_lista(cls):
        return cls.lista

    def to_dict(self):
        return {
            "nombre": self.nombre,
            "edad": int(self.edad),
            "contraseña": self.contra
        }    

## 2. La Lógica de Conexión a Supabase

Supabase se basa en **PostgreSQL**, una potente base de datos relacional. Antes de ejecutar esto, debes instalar la librería oficial de Supabase para Python ejecutando en la terminal: `pip install supabase`

**IMPORTANTE:**
1. Entra a supabase.com, crea un proyecto.
2. Ve a 'Table Editor' y crea una tabla llamada `usuarios`.
3. Añade las columnas `nombre` (text), `telefono` (text), `edad` (int4), `comida_favorita` (text).
4. Desactiva RLS ('Row Level Security') temporalmente para pruebas sencillas, o crea policies públicas.
5. Copia tu URL y API Key en las variables de abajo.

In [ ]:
from supabase import create_client, Client
from backend import *
# pip install supabase

# Reemplaza estos valores con la URL y Key (anon public) de tu proyecto en Supabase
SUPABASE_URL = "aqui va la url"
SUPABASE_KEY = "aqui va la key"

# Creamos el cliente global de Supabase
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

class GestorUsuarios:
    def __init__(self):
        self.lista_usuarios = []
        self.usuarios_nuevos = []  # Llevaremos rastro de los que insertemos en esta sesión
        
    def agregar_localmente(self, usuario):
        self.lista_usuarios.append(usuario)
        self.usuarios_nuevos.append(usuario)

    def cargar_desde_supabase(self):
        try:
            respuesta = supabase.table("usuarios").select("*").execute()
            datos = respuesta.data
            Usuario.lista = [] # Limpiamos la lista global del backend
            for val in datos:
                # Al instanciarlo, el __init__ del backend ya lo mete a Usuario.lista
                Usuario(val.get('nombre', ''), val.get('edad', 0), val.get('contraseña', ''))
            
            # Vaciamos usuarios_nuevos porque lo que acabamos de bajar ya está en la nube
            self.usuarios_nuevos = [] 
            print(f"Sincronizado: {len(Usuario.lista)} usuarios cargados.")
        except Exception as e:
            print("Error al cargar:", e)

    def guardar_en_supabase(self):
        """Ésta función se llamará al cerrar nuestra ventana visual"""
        if not self.usuarios_nuevos:
            print("No hay usuarios nuevos para enviar a Supabase.")
            return
            
        print(f"Subiendo {len(self.usuarios_nuevos)} nuevos usuarios a Supabase...")
        
        # Transformamos nuestra sub-lista de objetos nuevos a diccionarios
        lista_a_insertar = [u.to_dict() for u in self.usuarios_nuevos]
            
        try:
            # Un query '.insert()' manda a agregar varios registros en una sola operación (bulk insert)
            respuesta = supabase.table("usuarios").insert(lista_a_insertar).execute()
            print("¡Nuevos datos respaldados exitosamente a la base de datos remota!")
        except Exception as e:
            print("No se pudo guardar la información a Supabase:", e)

## 3. La Interfaz Gráfica (El Frontend con Tkinter)
Conectaremos la lógica de Supabase a los botones de Tkinter. Aprovecharemos nuevamente la capacidad de capturar cuando se cierra la ventana (`WM_DELETE_WINDOW`) para guardar exclusivamente los **nuevos registros** a nuestra tabla relacional en la nube.

In [ ]:
import tkinter as tk
from backend import *
from tkinter import messagebox

from basedatos import *

# 1. Instancia el gestor globalmente
gestor = GestorUsuarios()

def ventana_registro():
    venta1=tk.Tk()
    venta1.title("Base de datos")
    venta1.geometry("400x400")
    

    etiqueta1=tk.Label(venta1,text="Nombre")
    etiqueta1.pack()
    entrada1=tk.Entry(venta1,width=40)
    entrada1.pack(pady=15)

    etiqueta2=tk.Label(venta1,text="Edad")
    etiqueta2.pack()
    entrada2=tk.Entry(venta1,width=40)
    entrada2.pack(pady=15)

    etiqueta3=tk.Label(venta1,text="Contraseña")
    etiqueta3.pack()
    entrada3=tk.Entry(venta1,width=40)
    entrada3.pack(pady=15)
    
    def registrar():
        name=entrada1.get()
        age=entrada2.get()
        contra=entrada3.get()
        newuser=Usuario(name,age,contra)
        gestor.usuarios_nuevos.append(newuser)
        entrada1.delete(0,tk.END)
        entrada2.delete(0,tk.END)
        entrada3.delete(0,tk.END)
        messagebox.showinfo("Registro de usuario","Tu registro fué exitoso")


    boton1=tk.Button(venta1,text="Registrar",command=registrar)
    boton1.pack(pady=15)
    def mostrar():
        usuarios = Usuario.mostrar_lista()
        for u in usuarios:
            print(u.mostrar_datos())

    boton2=tk.Button(venta1,text="Mostrar lista",command=mostrar)
    boton2.pack(pady=15)

    def al_cerrar():
        print("Guardando datos antes de salir...")
        Usuario.guardar_usuarios()
        gestor.guardar_en_supabase()
        venta1.destroy() # Cierra la ventana físicamente
    
    # Configuración de que pasa al cerrar la ventana
    venta1.protocol("WM_DELETE_WINDOW", al_cerrar)

    venta1.mainloop()

def ventana_admin():
    ven3=tk.Tk()


def ventana_login():
    ven2=tk.Tk()
    ven2.title("Inicio de Sesión")
    ven2.geometry("400x300")
    gestor.cargar_desde_supabase()
    etiqueta3=tk.Label(ven2,text="Usuario")
    etiqueta3.pack()
    entrada4=tk.Entry(ven2,width=60)
    entrada4.pack(pady=10)
    etiqueta4=tk.Label(ven2,text="Password")
    etiqueta4.pack(pady=10)
    entrada5=tk.Entry(ven2,width=60)
    entrada5.pack(pady=10)

    def iniciar():
        name = entrada4.get()
        password = entrada5.get()

        # Buscamos si el usuario existe en la lista que bajamos de Supabase
        usuario_encontrado = None

        for u in Usuario.lista:
            if u.nombre == name:
                usuario_encontrado = u
                break

        # Validaciones
        if usuario_encontrado:
            if usuario_encontrado.contra == password:
                if name == "Administrador":
                    messagebox.showinfo("Login", "Bienvenido, Administrador")
                    ven2.destroy() # Cerramos login
                    ventana_registro() # Abrimos panel
                else:
                    messagebox.showinfo("Login", f"Bienvenido {name}")
                    # Aquí podrías abrir una ventana para usuarios normales si quisieras
            else:
                messagebox.showerror("Error", "Contraseña incorrecta")
        else:
            messagebox.showwarning("Login", "El usuario no existe en la base de datos")
    
    

    boton4=tk.Button(ven2,text="Iniciar sesión",command=iniciar)
    boton4.pack(pady=10)

    ven2.mainloop()

ventana_login()